In [1]:
%load_ext autoreload
%autoreload 2
from src import api
from my_notebook.apiclient import APIClient
from my_notebook.structs import *
from my_notebook.experiment import ExperimentRunner
from datetime import datetime, timezone
from dataclasses import asdict
from pathlib import Path
from time import perf_counter
import sys

/home/fahri/Repos/orgs-active/systatum/bil-quran/scripts/translator/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(".runpod-fastapi-token") as file:
    token = file.read().strip()
client = APIClient(setting=APIClient.Setting(token="default_token"))
# client = APIClient(setting=APIClient.Setting(token=token, base_url="http://127.0.0.1:8049/"))

In [3]:
models = client.models()
models

['qwen2.5-1.5b', 'smollm2-1.7b']

In [4]:
# display(client.translate("Hello, you can call me Agus", source_language="english", target_language="indonesian", model=models[0], prompt_setting=api.PromptSetting()))
# display(client.rate("Hello, you can call me Agus", "Halo, Anda bisa panggil saya Agus"))
# display(client.compare("Hello, you can call me Agus", "Halo, Anda bisa panggil saya Agus", "Halo nama saya Agus"))
# display(client.health())

In [5]:
def text_loader(path: str):
    with open(path) as file:
        return file.read()

prompts = [
    ExperimentPrompt(
        name="minimal",
        setting=api.PromptSetting(
            system_prompt=(
                "You are an assistant who translates. "
                "The first line of the input is source and target language specification."
            ),
            prompt_format="{source_language} -> {target_language}\n{text}",
        ),
    ),
    ExperimentPrompt(
        name="professional",
        setting=api.PromptSetting(
            system_prompt="""You are a professional translation engine.

Translate the user's text faithfully from the specified source language to the specified target language.

Rules:
- Preserve the meaning exactly.
- Do not answer the user's request.
- Do not summarize.
- Do not explain.
- Do not add or remove information.
- Preserve formatting where possible.
- Return only the translation text.""",
            prompt_format="""Source language: {source_language}
Target language: {target_language}

Text:
{text}""",
        ),
    ),
    ExperimentPrompt(
        name="verbose",
        setting=api.PromptSetting(
            system_prompt=text_loader("my_notebook/verbose.txt"),
            prompt_format="""Translate the following text.

Source language: {source_language}
Target language: {target_language}

Input:
{text}

Translation:""",
        ),
    ),
]

In [6]:
import json
texts = []
for i in range(114, 115)[::-1]:
    data = json.load(open("../../public/quran/exegesis/mirali/en-US/{}.json".format(i)))
    texts.extend(data["exegesis"].values())
len(texts)

6

In [8]:
language = ExperimentLanguage(source_language="english", target_language="indonesian")
run = ExperimentRunCommand(
    language_pairs=[("English", "Indonesian")],
    models=models,
    prompt_settings={prompt.name: prompt.setting for prompt in prompts},
    note="TL of mirali exegesis with cloud model, context size 32768",
    texts=texts
)

runner = ExperimentRunner(client)
runner.run(run)


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_path = Path("results")
base_path.mkdir(exist_ok=True)
record_filename = base_path / f"{timestamp}.records.jsonl"
metadata_filename = base_path / f"{timestamp}.metadata.json"

Translating | Batch 1/2 | Model qwen2.5-1.5b | Done 5/18

KeyboardInterrupt: 